In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("machine_type.csv", header=None)
df.columns = ["mtid", "type", "created_at", "updated_at"]

df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
df["updated_at"] = pd.to_datetime(df["updated_at"], utc=True)

# Standardise text — remove accidental spaces and force uppercase
df["type"] = df["type"].str.strip().str.upper()

df.to_csv("machine_type_preprocessed.csv", index=False)
print("Saved!")
print(df.shape)

Saved!
(3, 4)


In [ ]:
df_nongascut = pd.read_csv("summarize_nongascut_machine.csv", header=None)

df_nongascut.columns = [
    "business_date", "shift_name", "machine_type", "machine_name",
    "on_time", "off_time", "time_span",
    "total_lpg_cons", "total_heating_o2",
    "net_travel_in_mm", "mm_per_min"
]

before = len(df_nongascut)
df_nongascut.drop_duplicates(inplace=True)
after = len(df_nongascut)

df_nongascut["business_date"] = pd.to_datetime(df_nongascut["business_date"], utc=True)
df_nongascut["on_time"] = pd.to_datetime(df_nongascut["on_time"], utc=True)
df_nongascut["off_time"] = pd.to_datetime(df_nongascut["off_time"], utc=True)

df_nongascut["shift_name"] = df_nongascut["shift_name"].str.strip().str.upper()
df_nongascut["machine_type"] = df_nongascut["machine_type"].str.strip().str.upper()
df_nongascut["machine_name"] = df_nongascut["machine_name"].str.strip()

df_nongascut["mm_per_min_outlier_flag"] = df_nongascut["mm_per_min"] > 10000
df_nongascut["stationary_flag"] = df_nongascut["mm_per_min"] == 0

df_nongascut.to_csv("summarize_nongascut_preprocessed.csv", index=False)
print("Saved!")
print(df_nongascut.shape)


Saved!
(29547, 13)


In [ ]:
df_machines = pd.read_csv("machines.csv", header=None)
df_machines.columns = ["mid", "name", "hardware_id", "des", 
                        "msid", "mtid", "hid", "orgid", 
                        "mcsid", "mcid", "rpm_multiplication_factor", 
                        "notify", "deleted", "created_at", "updated_at"]

df_machines["notify"] = df_machines["notify"].map({"t": True, "f": False})


df_machines["deleted"] = df_machines["deleted"].map({"t": True, "f": False})

df_machines["des"] = df_machines["des"].str.strip().str.upper()
df_machines["des"] = df_machines["des"].replace("CLADDING", "CLAD")
df_machines["des"] = df_machines["des"].replace("GAS CUTTING MACHINE", "GASCUTTING")

df_machines["created_at"] = pd.to_datetime(df_machines["created_at"], utc=True)
df_machines["updated_at"] = pd.to_datetime(df_machines["updated_at"], utc=True)

df_machines.to_csv("machines_preprocessed.csv", index=False)
print("Saved!")
print(df_machines.shape)

Saved!
(17, 15)


In [16]:
df_user = pd.read_csv("user.csv", header=None)

df_user.columns = [
    "uid", "name", "email", "phno",
    "roleid", "hid", "orgid",
    "certificate_id", "identification_no",
    "password", "opid", "operator_rfid",
    "deleted", "created_at", "updated_at",
    "username", "current_session_token", "active_status",
    "csrf_token", "token_created_at",
    "csrf_token2", "token_created_at2"
]

df_user["phno"] = df_user["phno"].apply(lambda x: str(int(x)) if pd.notna(x) else None)

df_user["hid"] = pd.array(df_user["hid"], dtype=pd.Int64Dtype())

df_user["deleted"] = df_user["deleted"].map({"t": True, "f": False})
df_user["active_status"] = df_user["active_status"].map({"t": True, "f": False})

df_user["certificate_id"] = df_user["certificate_id"].replace("NIL", None)
df_user["identification_no"] = df_user["identification_no"].replace("NIL", None)

df_user["created_at"] = pd.to_datetime(df_user["created_at"], utc=True)
df_user["updated_at"] = pd.to_datetime(df_user["updated_at"], utc=True)
df_user["token_created_at"] = pd.to_datetime(df_user["token_created_at"], utc=True, errors="coerce")
df_user["token_created_at2"] = pd.to_datetime(df_user["token_created_at2"], utc=True, errors="coerce")

df_user["email"] = df_user["email"].str.strip().str.lower()

# Fill missing phone number with dummy value
df_user["phno"] = df_user["phno"].fillna("0000000000")

# Fill missing hid with 0 (no hardware assigned)
df_user["hid"] = df_user["hid"].fillna(0)

# Fill missing certificate_id with dummy value
df_user["certificate_id"] = df_user["certificate_id"].fillna("NOT PROVIDED")

# Fill missing identification_no with dummy value
df_user["identification_no"] = df_user["identification_no"].fillna("NOT PROVIDED")

# Fill missing operator_rfid with dummy value
df_user["operator_rfid"] = df_user["operator_rfid"].fillna("NOT ASSIGNED")

# Fix float values in operator_rfid
df_user["operator_rfid"] = df_user["operator_rfid"].apply(
    lambda x: str(int(float(x))) if x != "NOT ASSIGNED" and x is not None else x
)

# Fill missing username from email prefix
df_user["username"] = df_user.apply(
    lambda row: row["email"].split("@")[0] if pd.isna(row["username"]) else row["username"],
    axis=1
)
# Fill missing current_session_token with dummy value
df_user["current_session_token"] = df_user["current_session_token"].fillna("NO ACTIVE SESSION")

# Fill missing csrf tokens with dummy value
df_user["csrf_token"] = df_user["csrf_token"].fillna("NO ACTIVE SESSION")
df_user["csrf_token2"] = df_user["csrf_token2"].fillna("NO ACTIVE SESSION")

# Fill missing token timestamps with placeholder date
df_user["token_created_at"] = df_user["token_created_at"].fillna(pd.Timestamp("1970-01-01", tz="UTC"))
df_user["token_created_at2"] = df_user["token_created_at2"].fillna(pd.Timestamp("1970-01-01", tz="UTC"))

df_user.drop(columns=["opid"], inplace=True)

df_user.to_csv("user_preprocessed.csv", index=False)
print("Saved!")
print(df_user.shape)


Saved!
(24, 21)


In [22]:
df_dev = pd.read_csv("deviation.csv", header=None)
df_dev.columns = ["hardware_id", "oid", "shid", "start_tm", "end_tm", "span", "type", "parameter"]

before = len(df_dev)
df_dev.drop_duplicates(inplace=True)
after = len(df_dev)

df_dev["oid"] = df_dev["oid"].replace(0, None)

df_dev["start_tm"] = pd.to_datetime(df_dev["start_tm"], utc=True)
df_dev["end_tm"] = pd.to_datetime(df_dev["end_tm"], utc=True)

df_dev["hardware_id"] = df_dev["hardware_id"].str.strip().str.upper()
df_dev["type"] = df_dev["type"].str.strip().str.lower()
df_dev["parameter"] = df_dev["parameter"].str.strip().str.lower()

df_dev = pd.read_csv("deviation_preprocessed.csv")

# Fill oid nulls with 0
df_dev["oid"] = df_dev["oid"].fillna(0).astype(int)

df_dev.to_csv("deviation_preprocessed.csv", index=False)
print("Saved!")

Saved!


In [33]:
df_periodic = pd.read_csv("periodic_data_interval2.csv", low_memory=False)

df_periodic.drop(columns=["motor_cur", "motor_volt", "weight"], inplace=True)

df_periodic[["position_start", "position_end"]] = df_periodic["position"].str.split(",", expand=True).astype(float)
df_periodic.drop(columns=["position"], inplace=True)

flag_map = {0: "none", 1: "low", 2: "high"}
df_periodic["current_deviation_flag"] = df_periodic["current_deviation_flag"].map(flag_map)
df_periodic["voltage_deviation_flag"] = df_periodic["voltage_deviation_flag"].map(flag_map)
df_periodic["gas_deviation_flag"] = df_periodic["gas_deviation_flag"].map(flag_map)

df_periodic["business_date"] = pd.to_datetime(df_periodic["business_date"], utc=True)
df_periodic["tm"] = pd.to_datetime(df_periodic["tm"], utc=True)
df_periodic["created_at"] = pd.to_datetime(df_periodic["created_at"], utc=True)

df_periodic["machine_type"] = df_periodic["machine_type"].str.strip().str.upper()
df_periodic["machine_name"] = df_periodic["machine_name"].str.strip()
df_periodic["shift_name"] = df_periodic["shift_name"].str.strip().str.upper()
df_periodic["mstatus"] = df_periodic["mstatus"].str.strip().str.lower()
df_periodic["type"] = df_periodic["type"].str.strip().str.lower()
df_periodic["hardware_id"] = df_periodic["hardware_id"].str.strip().str.upper()

df_periodic["rpm"] = df_periodic["rpm"].clip(lower=0)

df_periodic.drop(columns=["dis"], inplace=True)

health_cols = ["health_status_lpg_flow_meter", "health_status_o2_flow_meter1", "health_status_o2_flow_meter2"]
for col in health_cols:
    df_periodic[col] = df_periodic[col].map({1.0: True, 0.0: False})

# df_periodic.to_csv("periodic_data_preprocessed.csv", index=False)
# print("Saved!")
# print(df_periodic.shape)

df_periodic = pd.read_csv("periodic_data_preprocessed.csv", low_memory=False)

# GMAW machines don't have gas columns - fill with 0
df_periodic["lpg_flow"] = df_periodic["lpg_flow"].fillna(0)
df_periodic["o2_flow_meter1"] = df_periodic["o2_flow_meter1"].fillna(0)
df_periodic["o2_flow_meter2"] = df_periodic["o2_flow_meter2"].fillna(0)
df_periodic["total_lpg_consumption"] = df_periodic["total_lpg_consumption"].fillna(0)
df_periodic["total_o2_consumption_meter1"] = df_periodic["total_o2_consumption_meter1"].fillna(0)
df_periodic["total_o2_consumption_meter2"] = df_periodic["total_o2_consumption_meter2"].fillna(0)
df_periodic["health_status_lpg_flow_meter"] = df_periodic["health_status_lpg_flow_meter"].fillna(False)
df_periodic["health_status_o2_flow_meter1"] = df_periodic["health_status_o2_flow_meter1"].fillna(False)
df_periodic["health_status_o2_flow_meter2"] = df_periodic["health_status_o2_flow_meter2"].fillna(False)

# GASCUTTING machines don't have weld columns - fill with 0
df_periodic["weld_cur"] = df_periodic["weld_cur"].fillna(0)
df_periodic["weld_volt"] = df_periodic["weld_volt"].fillna(0)
df_periodic["weld_gas"] = df_periodic["weld_gas"].fillna(0)
df_periodic["hs_temp"] = df_periodic["hs_temp"].fillna(0)
df_periodic["amb_temp"] = df_periodic["amb_temp"].fillna(0)
df_periodic["rpm"] = df_periodic["rpm"].fillna(0)

# Position columns - only gas cutting machines have these
df_periodic["position_start"] = df_periodic["position_start"].fillna(0)
df_periodic["position_end"] = df_periodic["position_end"].fillna(0)
df_periodic["thickness"] = df_periodic["thickness"].fillna(0)
df_periodic["cut_mm_mtr"] = df_periodic["cut_mm_mtr"].fillna(0)
df_periodic["travel_in_mm"] = df_periodic["travel_in_mm"].fillna("0")

# Verify
# print("Total nulls remaining:", df_periodic.isnull().sum().sum())
# print(df_periodic.isnull().sum()[df_periodic.isnull().sum() > 0])

# Fix oid nulls
df_periodic["oid"] = df_periodic["oid"].fillna("0")

# Fix network nulls - 0 means no network signal
df_periodic["network"] = df_periodic["network"].fillna(0)

# Fix the FutureWarning for health status columns
df_periodic["health_status_lpg_flow_meter"] = df_periodic["health_status_lpg_flow_meter"].fillna(False).infer_objects(copy=False)
df_periodic["health_status_o2_flow_meter1"] = df_periodic["health_status_o2_flow_meter1"].fillna(False).infer_objects(copy=False)
df_periodic["health_status_o2_flow_meter2"] = df_periodic["health_status_o2_flow_meter2"].fillna(False).infer_objects(copy=False)

# Verify
# print("Total nulls remaining:", df_periodic.isnull().sum().sum())

# df_periodic.to_csv("periodic_data_preprocessed.csv", index=False)
# print("Saved!")
# print(df_periodic.shape)

df_periodic = pd.read_csv("periodic_data_preprocessed.csv", low_memory=False)
print(df_periodic.isnull().sum()[df_periodic.isnull().sum() > 0])


Series([], dtype: int64)


In [ ]:
df_md = pd.read_csv("machine_derived.csv", header=None)

df_md.columns = [
    "business_date", "shift_name", "machine_type", "machine_name",
    "oid", "period_start", "period_end",
    "active", "idle", "inrepair",
    "avg_weld_cur", "avg_weld_cur_deviation",
    "avg_weld_volt", "avg_weld_volt_deviation",
    "breakdown", "target_arc_time", "deposit"
]

df_md["oid"] = df_md["oid"].replace(0, None)

df_md["business_date"] = pd.to_datetime(df_md["business_date"], utc=True)

df_md["shift_name"] = df_md["shift_name"].str.strip().str.upper()
df_md["machine_type"] = df_md["machine_type"].str.strip().str.upper()
df_md["machine_name"] = df_md["machine_name"].str.strip()

df_md["machine_name"] = df_md["machine_name"].str.replace(" ", "")

df_md = pd.read_csv("machine_derived_preprocessed.csv")

# Fill oid nulls with 0
df_md["oid"] = df_md["oid"].fillna(0).astype(int)

print(df_md["oid"].isnull().sum())  # should show 0

df_md.to_csv("machine_derived_preprocessed.csv", index=False)
print("Saved!")

0
Saved!


In [26]:
df_gas = pd.read_csv("summarize_gascutting_machine.csv", header=None)

df_gas.columns = [
    "business_date", "shift_name", "machine_type", "machine_name",
    "on_time", "off_time", "time_span",
    "net_travel_in_mm", "net_lpg_consumption",
    "net_o2_consumption_meter1", "net_o2_consumption_meter2",
    "mm_per_min", "thickness", "cut_mm_mtr"
]

df_gas["business_date"] = pd.to_datetime(df_gas["business_date"], utc=True)
df_gas["on_time"] = pd.to_datetime(df_gas["on_time"], utc=True)
df_gas["off_time"] = pd.to_datetime(df_gas["off_time"], utc=True)

df_gas["shift_name"] = df_gas["shift_name"].str.strip().str.upper()
df_gas["machine_type"] = df_gas["machine_type"].str.strip().str.upper()
df_gas["machine_name"] = df_gas["machine_name"].str.strip()

df_gas["mm_per_min_outlier_flag"] = df_gas["mm_per_min"] > 10000
df_gas["no_gas_data_flag"] = (df_gas["net_lpg_consumption"] == 0) & (df_gas["net_o2_consumption_meter1"] == 0)

# df_gas.to_csv("summarize_gascutting_preprocessed.csv", index=False)
# print("Saved!")
# print(df_gas.shape)

df_gas = pd.read_csv("summarize_gascutting_preprocessed.csv")
df_gas["thickness"] = df_gas["thickness"].fillna(0)
df_gas["cut_mm_mtr"] = df_gas["cut_mm_mtr"].fillna(0)
print("summarize_gascutting nulls:", df_gas.isnull().sum().sum())
df_gas.to_csv("summarize_gascutting_preprocessed.csv", index=False)
print("summarize_gascutting saved!")

summarize_gascutting nulls: 0
summarize_gascutting saved!


In [125]:
df_nongascut = pd.read_csv("summarize_nongascut_machine.csv", header=None)

df_nongascut.columns = [
    "business_date", "shift_name", "machine_type", "machine_name",
    "on_time", "off_time", "time_span",
    "total_lpg_cons", "total_heating_o2",
    "net_travel_in_mm", "mm_per_min"
]

before = len(df_nongascut)
df_nongascut.drop_duplicates(inplace=True)
after = len(df_nongascut)

df_nongascut["business_date"] = pd.to_datetime(df_nongascut["business_date"], utc=True)
df_nongascut["on_time"] = pd.to_datetime(df_nongascut["on_time"], utc=True)
df_nongascut["off_time"] = pd.to_datetime(df_nongascut["off_time"], utc=True)

df_nongascut["shift_name"] = df_nongascut["shift_name"].str.strip().str.upper()
df_nongascut["machine_type"] = df_nongascut["machine_type"].str.strip().str.upper()
df_nongascut["machine_name"] = df_nongascut["machine_name"].str.strip()

df_nongascut["mm_per_min_outlier_flag"] = df_nongascut["mm_per_min"] > 10000
df_nongascut["stationary_flag"] = df_nongascut["mm_per_min"] == 0

df_nongascut.to_csv("summarize_nongascut_preprocessed.csv", index=False)
print("Saved!")
print(df_nongascut.shape)


Saved!
(29547, 13)


In [ ]:
df_clad = pd.read_csv("summarize_clad_details_info.csv", header=None)

df_clad.drop(columns=[8, 9, 11, 12, 14, 15], inplace=True)

df_clad.columns = [
    "business_date", "shift_name", "oid", "machine_type", "machine_name",
    "ontime", "offtime", "time_span",
    "avg_weld_cur", "avg_weld_volt", "loss_weight"
]

before = len(df_clad)
df_clad.drop_duplicates(inplace=True)
after = len(df_clad)
# print(f"Duplicates removed: {before - after}")

df_clad["oid"] = df_clad["oid"].replace(0, None)

df_clad["business_date"] = pd.to_datetime(df_clad["business_date"], utc=True)
df_clad["ontime"] = pd.to_datetime(df_clad["ontime"], utc=True)
df_clad["offtime"] = pd.to_datetime(df_clad["offtime"], utc=True)

df_clad["shift_name"] = df_clad["shift_name"].str.strip().str.upper()
df_clad["machine_type"] = df_clad["machine_type"].str.strip().str.upper()
df_clad["machine_name"] = df_clad["machine_name"].str.strip()

df_clad["loss_weight_flag"] = df_clad["loss_weight"] < 0
df_clad["loss_weight"] = df_clad["loss_weight"].clip(lower=0)
df_clad["zero_duration_flag"] = df_clad["time_span"] == "00:00:00"

# df_clad.to_csv("summarize_clad_preprocessed.csv", index=False)
# print("Saved!")
# print(df_clad.shape)


df_clad = pd.read_csv("summarize_clad_preprocessed.csv")
df_clad["oid"] = df_clad["oid"].fillna(0).astype(int)
print("summarize_clad oid nulls:", df_clad["oid"].isnull().sum())
df_clad.to_csv("summarize_clad_preprocessed.csv", index=False)
print("summarize_clad saved!")

summarize_clad oid nulls: 0
summarize_clad saved!


In [34]:
import pandas as pd

files = {
    "machine_type": "machine_type_preprocessed.csv",
    "machines": "machines_preprocessed.csv",
    "deviation": "deviation_preprocessed.csv",
    "machine_derived": "machine_derived_preprocessed.csv",
    "summarize_clad": "summarize_clad_preprocessed.csv",
    "summarize_gascutting": "summarize_gascutting_preprocessed.csv",
    "summarize_nongascut": "summarize_nongascut_preprocessed.csv",
    "periodic_data": "periodic_data_preprocessed.csv",
}

for name, file in files.items():
    df = pd.read_csv(file, low_memory=False)
    nulls = df.isnull().sum().sum()
    if nulls > 0:
        print(f"\n{name}: {nulls} total nulls")
        print(df.isnull().sum()[df.isnull().sum() > 0])
    else:
        print(f"{name}: ✅ No nulls")

machine_type: ✅ No nulls
machines: ✅ No nulls
deviation: ✅ No nulls
machine_derived: ✅ No nulls
summarize_clad: ✅ No nulls
summarize_gascutting: ✅ No nulls
summarize_nongascut: ✅ No nulls
periodic_data: ✅ No nulls
